In [1]:
import torch
import torch.nn as nn

In [2]:
class SeqToVecGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, last_state = self.gru(X)
        return self.output(outputs[:, -1])

In [3]:
class SeqToSeqGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, last_state = self.gru(X)
        return self.output(outputs)

In [4]:
class VecToSeqGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers, seq_len):
        super().__init__()
        self.seq_len = seq_len
        self.gru = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x_vec):
        X_repeated = x_vec.unsqueeze(1).repeat(1, self.seq_len, 1)
        outputs, last_state = self.gru(X_repeated)
        return self.output(outputs)

In [5]:
class EncoderDecoderGRU(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers, seq_len):
        super().__init__()
        self.seq_len = seq_len
        self.hidden_size = hidden_size
        self.encoder = nn.GRU(input_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.decoder = nn.GRU(hidden_size, hidden_size, num_layers=num_layers, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        batch_size = X.shape[0]
        _, last_state = self.encoder(X)
        decoder_input = torch.zeros(batch_size, self.seq_len, self.hidden_size)
        outputs, _ = self.decoder(decoder_input, last_state)
        return self.output(outputs)